# 01 – EDA and Modeling DecisionsThis notebook is a **template** for exploratory data analysis (EDA) and key modeling decisionsfor the *Credit Card Fraud Detection* dataset (Kaggle: `mlg-ulb/creditcardfraud`).> Fill in this template once you have downloaded the dataset to `data/raw/`.

## 1. Problem Framing and Class Imbalance- **Goal**: Detect fraudulent credit card transactions as anomalies.- **Setting**: Highly imbalanced binary classification / anomaly detection.- **Target**: `Class` (1 = fraud, 0 = non-fraud).- **Key facts** (to confirm via data):  - Number of rows.  - Number and fraction of fraud cases.  - Class imbalance ratio.In this section, quantify the imbalance and discuss why metrics like accuracy are misleading.

## 2. Data DictionaryDocument the features available in the dataset:- `Time`: Seconds elapsed between this transaction and the first transaction in the dataset.- `Amount`: Transaction amount.- `Class`: Target label (1 = fraud, 0 = non-fraud).- `V1`–`V28`: Numeric features obtained via PCA transformation of original confidential inputs.Discuss implications:- PCA components limit direct interpretability.- `Time` is useful for leakage-aware splitting and temporal analysis but is **not** a customer ID.

## 3. Leakage-Safe Split Plan (Time-Based)We aim to simulate a realistic scenario where models trained on **past** data are evaluated on **future** data.Plan:- Explore distribution of `Time` and potential breakpoints.- Choose a time-based rule to split data into train / validation / test, e.g.:  - Train: earliest 60–70% of time range.  - Validation: next 10–20%.  - Test: final 10–20%.- Ensure that all metrics are reported on **held-out future data**.Use this section to:- Visualize class imbalance over time.- Justify chosen split boundaries.

## 4. Preprocessing DecisionsKey questions:- How to scale `Amount` and `Time` (e.g., `StandardScaler`, `RobustScaler`, log-transform)?- Do we need any transformations on `V1`–`V28` beyond scaling (likely not, as they are PCA components)?- How do we handle missing values (if any)?Draft decisions (to refine with EDA):- Apply scaling to `Amount` and possibly `Time` using a robust or standard scaler fit on the training set.- Keep `V1`–`V28` as-is but confirm their distributions.- Avoid oversampling/SMOTE in the first iteration; rely on thresholding and metrics.

## 5. Model Plan (Six Models + Ensemble)We plan to build six models in total, including anomaly detection methods and an ensemble.The exact set can be refined, but we explicitly **swap LSTM-AE with COPOD** due to the lack of true sequences.Outline candidate models:- Unsupervised / semi-supervised anomaly detectors (e.g., Isolation Forest, One-Class SVM, **COPOD via PyOD**).- Supervised baseline(s) using the `Class` label (e.g., logistic regression, tree-based model).- Ensemble model combining scores from base models.For each model, capture:- Intuition and expected behavior.- Strengths and weaknesses for this dataset.- Computational cost and suitability for free-tier deployments.

## 6. Metrics Plan (PR-AUC and Thresholding)Given the class imbalance, metrics will focus on the positive (fraud) class:- **PR-AUC (Precision–Recall AUC)** as the primary model selection metric.- **ROC-AUC** as a secondary check.- **Precision, Recall, F1** at selected thresholds.Thresholding strategy:- Explore precision–recall curves for each model.- Consider thresholds that:  - Maximize F1, and  - Achieve minimum acceptable precision at high recall (e.g., recall ≥ 0.8, precision ≥ 0.1).Document choices and trade-offs in this section.

## 7. Feature Importance / ExplainabilityBecause most features (`V1`–`V28`) are PCA components, they are **not directly interpretable**.This limits classic feature importance narratives.Proposed strategies:- Use **permutation importance** on a supervised proxy model (e.g., gradient boosting or logistic regression).- Use **model-agnostic sensitivity analysis**:  - Perturb individual features (or groups of features) and observe changes in predicted scores.- Consider simple SHAP/feature impact plots on the supervised model, with clear caveats about PCA.Use this section to experiment and capture how we will communicate model behavior despite PCA-based features.

## 8. ReproducibilityTo ensure experiments are reproducible:- Fix random seeds for:  - NumPy, Python's `random`, and any model-specific seeds.- Record environment details:  - Python version (3.10+).  - Versions of key libraries: `pandas`, `numpy`, `scikit-learn`, `pyod`, etc.- Document data paths and any filtering/exclusion rules.In this section, include cells to:- Print package versions.- Set seeds.

In [ ]:
"""Minimal imports and placeholders.Run this after you have the dataset in `data/raw/`."""import pandas as pdimport numpy as npDATA_PATH = "data/raw/creditcard.csv"  # Adjust if neededprint("TODO: load data from data/raw once available.")# Example (uncomment when file exists):# df = pd.read_csv(DATA_PATH)# display(df.head())